### modelos

Teste(s): modelos monolíticos; ensembles de árvores.

Objetivo(s): desempenho; redução de latência de inferência; manutenção.

In [ ]:
Decision Tree (Árvore de Decisão) — Modelo Selecionado.   Random Forest   MLP - Multi-Layer Perceptron (8,4)   Regressão Logística

In [1]:
import time
import pickle
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

# 1. GERAÇÃO DE DATASET SINTÉTICO (Simulando Sensores com diferentes Escalas e Correlação)
def generate_sensor_data(n_samples=2000, seed=42):
    np.random.seed(seed)
    # Features base: Temp (20-40), Umidade (30-90), Sensor_IR (0 ou 1)
    temp = np.random.uniform(20, 40, n_samples)
    umidade = np.random.uniform(30, 90, n_samples)
    ir = np.random.choice([0, 1], size=n_samples)

    # Feature com escala discrepante e forte correlação com temperatura (ex: Pressão/Ponto de Orvalho)
    pressao_hpa = 1013.25 - (temp * 1.5) + np.random.normal(0, 0.5, n_samples) # Escala ~1000

    X = pd.DataFrame({
        'temperatura': temp,
        'umidade': umidade,
        'pressao_hpa': pressao_hpa,
        'presenca_ir': ir
    })

    # Regra de Rótulo (4 classes: Normal, Presença, Alerta, Crítico)
    y = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        if X.loc[i, 'temperatura'] >= 35 or X.loc[i, 'umidade'] >= 85:
            y[i] = 3 # Critico
        elif X.loc[i, 'temperatura'] >= 30 or X.loc[i, 'umidade'] >= 75:
            y[i] = 2 # Alerta
        elif X.loc[i, 'presenca_ir'] == 1:
            y[i] = 1 # Presenca
        else:
            y[i] = 0 # Normal

    return X, y

# 2. INTRODUÇÃO DE DATA DRIFT
def apply_data_drift(X, drift_factor=0.2):
    X_drift = X.copy()
    # Simula descalibração do sensor de temp (+3°C) e degradação na umidade (+15%)
    X_drift['temperatura'] += 3.0
    X_drift['umidade'] += 15.0
    # Adiciona ruído na pressão
    X_drift['pressao_hpa'] += np.random.normal(5.0, 2.0, len(X_drift))
    return X_drift

# 3. SETUP DOS MODELOS
models = {
    'Baseline (Most Frequent)': DummyClassifier(strategy='most_frequent'),
    'Decision Tree (Max Depth 4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest (32 Trees)': RandomForestClassifier(n_estimators=32, max_depth=4, random_state=42),
    'MLP Classifier (8,4)': MLPClassifier(hidden_layer_sizes=(8, 4), max_iter=500, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
}

# 4. EXECUÇÃO DO BENCHMARK
X, y = generate_sensor_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test_drift = apply_data_drift(X_test)

results = []

for name, model in models.items():
    # Treinamento
    model.fit(X_train, y_train)

    # 1. Medição de Tamanho em Memória (Bytes via Pickle)
    model_bytes = len(pickle.dumps(model))

    # 2. Medição de F1-Score Sem Drift
    y_pred = model.predict(X_test)
    f1_clean = f1_score(y_test, y_pred, average='macro')

    # 3. Medição de F1-Score COM Data Drift
    y_pred_drift = model.predict(X_test_drift)
    f1_drift = f1_score(y_test, y_pred_drift, average='macro')

    # 4. Medição de Latência Média por Inferência (em us)
    start_time = time.perf_counter()
    n_runs = 1000
    for _ in range(n_runs):
        _ = model.predict(X_test.iloc[:1])
    end_time = time.perf_counter()
    latency_us = ((end_time - start_time) / n_runs) * 1e6

    # Registro dos Resultados
    results.append({
        'Modelo': name,
        'Tamanho Estimado (Bytes)': model_bytes,
        'Latência (µs/amostra)': round(latency_us, 2),
        'F1-Score (Sem Drift)': round(f1_clean, 3),
        'F1-Score (Com Drift)': round(f1_drift, 3),
        'Queda por Drift (%)': round(((f1_clean - f1_drift) / f1_clean) * 100, 2)
    })

# 5. TABELA COMPARATIVA
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                     Modelo  Tamanho Estimado (Bytes)  Latência (µs/amostra)  F1-Score (Sem Drift)  F1-Score (Com Drift)  Queda por Drift (%)
   Baseline (Most Frequent)                       648                 519.37                 0.120                 0.120                 0.00
Decision Tree (Max Depth 4)                      2546                2905.73                 0.965                 0.616                36.14
   Random Forest (32 Trees)                     70341                9023.52                 0.995                 0.559                43.87
       MLP Classifier (8,4)                     12431                1083.83                 0.120                 0.120                 0.00
        Logistic Regression                      1024                1012.24                 0.759                 0.530                30.15
